In [72]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import pickle

import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from datetime import datetime
from pandas.tseries.offsets import MonthEnd

In [73]:
os.listdir('/data/aman_singh/acuuracy_check')

['missing_keys_drm.csv',
 'Heuristics_all_combination_qcom_cp_apr_live.xlsx',
 'seasonality_all2.csv',
 'chek_nan.csv',
 'missing_keys_drm2.csv',
 'all_combination_ecom_may_pred.csv',
 'QCOM Chain FC PSKU Primary_as_on_11th_Feb_2026.xlsb',
 'soh_recent_qcom.csv',
 'combine_model+missing_forecasts_brand_asm.ipynb',
 'April-26 Plans.xlsx',
 'QCOM Chain PSKU OTP Output',
 'Heuristics_all_combination_ecom_may_live.xlsx',
 'Norms 202602.csv',
 'duplicates_after_realignment.csv',
 'ALL Channels Accuracy_fva.ipynb',
 "mt_channels Live Run may'26.csv",
 'all_combination_qcom_cp_july_pred.csv',
 't_thres_df_2.csv',
 'ecom_chain_psku_offtake_to_secondary_v6_PROD.ipynb',
 'qcom_chain_depot_psku_primary_forecast.csv',
 "gt_channels Live Run may'26.csv",
 'seasonality.xlsx',
 'ECOM Chain PSKU Primary_as_on_12_Jan_2026 (1).xlsb',
 'trend_file_train_till_31_May_2026 (1).csv',
 'QCOM_NORMS_FINAL_offtake_to_secondary_chain_depot_psku.ipynb',
 'soh_base_may_run.csv',
 'combine_model+missing_forecasts.ip

In [74]:
base_dir = '/data/aman_singh/acuuracy_check'
input_table = 'TRN_DF_QCOM_OFFTAKE_CHAIN_PSKU'
run_month_inp = '2026-06-30'

In [75]:
def list_all_files_in_directory(root):
    out = []

    for path, subdirs, files in os.walk(root):
        for name in files:
            out.append(os.path.join(path, name))

    return out

In [76]:
list_all_files_in_directory(base_dir)

['/data/aman_singh/acuuracy_check/missing_keys_drm.csv',
 '/data/aman_singh/acuuracy_check/Heuristics_all_combination_qcom_cp_apr_live.xlsx',
 '/data/aman_singh/acuuracy_check/seasonality_all2.csv',
 '/data/aman_singh/acuuracy_check/chek_nan.csv',
 '/data/aman_singh/acuuracy_check/missing_keys_drm2.csv',
 '/data/aman_singh/acuuracy_check/all_combination_ecom_may_pred.csv',
 '/data/aman_singh/acuuracy_check/QCOM Chain FC PSKU Primary_as_on_11th_Feb_2026.xlsb',
 '/data/aman_singh/acuuracy_check/soh_recent_qcom.csv',
 '/data/aman_singh/acuuracy_check/combine_model+missing_forecasts_brand_asm.ipynb',
 '/data/aman_singh/acuuracy_check/April-26 Plans.xlsx',
 '/data/aman_singh/acuuracy_check/Heuristics_all_combination_ecom_may_live.xlsx',
 '/data/aman_singh/acuuracy_check/Norms 202602.csv',
 '/data/aman_singh/acuuracy_check/duplicates_after_realignment.csv',
 '/data/aman_singh/acuuracy_check/ALL Channels Accuracy_fva.ipynb',
 "/data/aman_singh/acuuracy_check/mt_channels Live Run may'26.csv",


In [77]:
def discover_channel(file_path):
    # file_path = file_path.split('/')

    # if 'ECOM' in file_path:
    #     return 'ECOM'
    # elif 'QCOM' in file_path:
    #     return 'QCOM'
    # elif 'MT' in file_path:
    #     return 'MT'
    # else:
    #     return 'Channel not found'

    return 'ECOM'


In [78]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [79]:
data_query = f"""
    select * from {input_table}
    where month_date >= '2023-01-31' and run_month = '{run_month_inp}'
        and platform_name in ('blinkit', 'swiggy', 'zepto')
"""

offtake_df = pd.read_sql(data_query, dev_conn)
offtake_df.head()

,MONTH_DATE,KEY,PLATFORM_NAME,PARENT_MATERIAL_CODE,BRAND_CODE,VOL_IN_RUM,RUN_MONTH,IMPUTED
0,2023-01-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,22.584,2026-06-30,0
1,2023-02-28,blinkit_718288,blinkit,718288.0,SAFF GOLD,17.568,2026-06-30,0
2,2023-03-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,25.332,2026-06-30,0
3,2023-04-30,blinkit_718288,blinkit,718288.0,SAFF GOLD,21.888,2026-06-30,0
4,2023-05-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,15.860,2026-06-30,0


In [80]:
offtake_df.columns = offtake_df.columns.str.lower()

In [81]:
offtake_df.duplicated(
    subset=['platform_name','parent_material_code', 'month_date']).sum()

0

In [82]:
offtake_df['brand_code'] = np.where(
    ((offtake_df['parent_material_code'] == 715096) &
    (offtake_df['brand_code'] == 'CO_SO_PCP')),
    'CO_SO_FS',
    offtake_df['brand_code']
)

In [83]:
# offtake_df = offtake_df[offtake_df['platform_name'].isin(
#     ['Amazon', 'Big Basket', 'Flipkart Grocery', 'Flipkart National'])]

In [84]:
offtake_df

,month_date,key,platform_name,parent_material_code,brand_code,vol_in_rum,run_month,imputed
0,2023-01-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,22.584,2026-06-30,0
1,2023-02-28,blinkit_718288,blinkit,718288.0,SAFF GOLD,17.568,2026-06-30,0
2,2023-03-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,25.332,2026-06-30,0
3,2023-04-30,blinkit_718288,blinkit,718288.0,SAFF GOLD,21.888,2026-06-30,0
4,2023-05-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,15.860,2026-06-30,0
...,...,...,...,...,...,...,...,...
31829,2027-03-31,zepto_810685,zepto,810685.0,SAF-MUSLI,0.000,2026-06-30,0
31830,2027-03-31,zepto_810738,zepto,810738.0,PABABY_GM,0.000,2026-06-30,0
31831,2027-03-31,zepto_810971,zepto,810971.0,PA_ESS_HO,0.000,2026-06-30,0
31832,2027-03-31,zepto_811005,zepto,811005.0,PA_ESS_HO,0.000,2026-06-30,0


In [85]:
offtake_df['key'] = offtake_df[['platform_name','parent_material_code']].astype(str).agg('_'.join, axis=1)
# offtake_df.rename(columns={'realigned_psku': 'parent_material_code'}, inplace=True)
# offtake_df.drop([ 'run_month'], axis=1, inplace=True)
offtake_df['parent_material_code'] = offtake_df['parent_material_code'].astype(int)

In [86]:
offtake_df.duplicated(subset=['key', 'month_date']).sum()

0

In [87]:
(offtake_df['key'] == offtake_df[['platform_name','parent_material_code']].astype(str).agg('_'.join, axis=1)).all()

False

In [88]:
# realigned_df.to_csv('OT_data_debug.csv', index=False)

### Collate MIL

In [89]:
base_dir

'/data/aman_singh/acuuracy_check'

In [90]:
def collate_file(file_hint, extension='.csv'):
    collated_file = pd.DataFrame()

    run_path = f'{base_dir}'
    all_files = list_all_files_in_directory(run_path)

    for file_path in all_files:
        if file_hint in file_path:
            if extension == '.csv':
                print(file_path)
                read_file = pd.read_csv(file_path)
                # read_file['channel'] = discover_channel(file_path)
                read_file['run'] = 'run'
                read_file['step'] = file_path.split('/')[3]
                read_file['file_path'] = file_path

                collated_file = pd.concat(
                    [collated_file, read_file]
                )
                del read_file

    return collated_file

In [91]:
trend_file_df = collate_file('trend_file_train_till')
prophet_file_df = collate_file('prophet_data_train_till')

/data/aman_singh/acuuracy_check/trend_file_train_till_31_May_2026 (1).csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_31_May_2026 (1).csv


In [92]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'parent_material_code', 'platform_name', 'vol_in_rum',
       'brand_code', 'qtr_ind_rate', 'vol_in_rum_value', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path'],
      dtype='object')

In [93]:
# forecast_train_till_file_df = collate_file('forecast_train_till_')

In [94]:
# forecast_train_till_file_df

In [95]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,brand_code,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path
0,blinkit_718288.0,2023-01-31,21.828000,21.337833,14.141480,23.283180,0.303115,0.296308,0.196376,0.323322,...,SAFF GOLD,138865.260689,0.313613,22.584,0.313613,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
1,blinkit_718288.0,2023-02-28,21.828000,21.337833,15.747385,21.731667,0.303115,0.296308,0.218676,0.301777,...,SAFF GOLD,138865.260689,0.243958,17.568,0.243958,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
2,blinkit_718288.0,2023-03-31,21.828000,21.337833,26.868887,25.585143,0.303115,0.296308,0.373116,0.355289,...,SAFF GOLD,138865.260689,0.351773,25.332,0.351773,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
3,blinkit_718288.0,2023-04-30,21.828000,21.337833,10.384195,23.536022,0.303115,0.296308,0.144200,0.326834,...,SAFF GOLD,138865.260689,0.303948,21.888,0.303948,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
4,blinkit_718288.0,2023-05-31,21.596000,21.337833,12.988040,21.183205,0.299893,0.296308,0.180359,0.294161,...,SAFF GOLD,138865.260689,0.220240,15.860,0.220240,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26442,zepto_810439.0,2026-09-30,0.827633,0.892150,0.274929,0.310768,0.026113,0.028149,0.008674,0.009805,...,SAF-MUSLI,315513.490535,0.000000,0.000,0.000000,2026-05-31,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
26443,zepto_810439.0,2026-10-31,0.827633,0.892150,0.231432,0.302197,0.026113,0.028149,0.007302,0.009535,...,SAF-MUSLI,315513.490535,0.000000,0.000,0.000000,2026-05-31,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
26444,zepto_810439.0,2026-11-30,0.827633,0.892150,0.411081,0.354295,0.026113,0.028149,0.012970,0.011178,...,SAF-MUSLI,315513.490535,0.000000,0.000,0.000000,2026-05-31,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
26445,zepto_810439.0,2026-12-31,0.827633,0.892150,0.258753,0.331626,0.026113,0.028149,0.008164,0.010463,...,SAF-MUSLI,315513.490535,0.000000,0.000,0.000000,2026-05-31,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...


In [96]:
# prophet_file_df.to_csv('Prophet_file_OT_FK_AZ_BB_.csv', index=False)

In [97]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'parent_material_code', 'platform_name', 'vol_in_rum',
       'brand_code', 'qtr_ind_rate', 'vol_in_rum_value', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path'],
      dtype='object')

In [98]:
trend_file_df.dtypes

key                          object
month_date                   object
pred_p3m                    float64
pred_p6m                    float64
pred_prophet                float64
pred_rf                     float64
pred_value_p3m              float64
pred_value_p6m              float64
pred_value_prophet          float64
pred_value_rf               float64
parent_material_code        float64
platform_name                object
vol_in_rum                  float64
brand_code                   object
qtr_ind_rate                float64
vol_in_rum_value            float64
vol_in_rum_treated          float64
vol_in_rum_value_treated    float64
train_till                   object
cov                         float64
run                          object
step                         object
file_path                    object
dtype: object

In [99]:
trend_file_df['month_date'] = pd.to_datetime(trend_file_df['month_date'])
prophet_file_df['month_date'] = pd.to_datetime(prophet_file_df['month_date'])

trend_file_df['train_till'] = pd.to_datetime(trend_file_df['train_till'])
prophet_file_df['train_till'] = pd.to_datetime(prophet_file_df['train_till'])

trend_file_df['run_month'] = pd.to_datetime(trend_file_df['train_till'] + MonthEnd(1))
prophet_file_df['run_month'] = pd.to_datetime(prophet_file_df['train_till'] + MonthEnd(1))

In [100]:
trend_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum(), \
prophet_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum()

(0, 0)

In [101]:
mappings = {}

for run_month in trend_file_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-06-30 00:00:00'): {Timestamp('2026-06-30 00:00:00'): 'M',
  Timestamp('2026-07-31 00:00:00'): 'M+1',
  Timestamp('2026-08-31 00:00:00'): 'M+2',
  Timestamp('2026-09-30 00:00:00'): 'M+3',
  Timestamp('2026-10-31 00:00:00'): 'M+4',
  Timestamp('2026-11-30 00:00:00'): 'M+5',
  Timestamp('2026-12-31 00:00:00'): 'M+6',
  Timestamp('2027-01-31 00:00:00'): 'M+7',
  Timestamp('2027-02-28 00:00:00'): 'M+8'}}

In [102]:
trend_file_df['M month'] = trend_file_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

In [103]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month
0,blinkit_718288.0,2023-01-31,21.828000,21.337833,14.141480,23.283180,0.303115,0.296308,0.196376,0.323322,...,0.313613,22.584,0.313613,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
1,blinkit_718288.0,2023-02-28,21.828000,21.337833,15.747385,21.731667,0.303115,0.296308,0.218676,0.301777,...,0.243958,17.568,0.243958,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
2,blinkit_718288.0,2023-03-31,21.828000,21.337833,26.868887,25.585143,0.303115,0.296308,0.373116,0.355289,...,0.351773,25.332,0.351773,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
3,blinkit_718288.0,2023-04-30,21.828000,21.337833,10.384195,23.536022,0.303115,0.296308,0.144200,0.326834,...,0.303948,21.888,0.303948,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
4,blinkit_718288.0,2023-05-31,21.596000,21.337833,12.988040,21.183205,0.299893,0.296308,0.180359,0.294161,...,0.220240,15.860,0.220240,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26442,zepto_810439.0,2026-09-30,0.827633,0.892150,0.274929,0.310768,0.026113,0.028149,0.008674,0.009805,...,0.000000,0.000,0.000000,2026-05-31,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3
26443,zepto_810439.0,2026-10-31,0.827633,0.892150,0.231432,0.302197,0.026113,0.028149,0.007302,0.009535,...,0.000000,0.000,0.000000,2026-05-31,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+4
26444,zepto_810439.0,2026-11-30,0.827633,0.892150,0.411081,0.354295,0.026113,0.028149,0.012970,0.011178,...,0.000000,0.000,0.000000,2026-05-31,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+5
26445,zepto_810439.0,2026-12-31,0.827633,0.892150,0.258753,0.331626,0.026113,0.028149,0.008164,0.010463,...,0.000000,0.000,0.000000,2026-05-31,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+6


In [104]:
trend_file_df[trend_file_df['M month'].notna()]

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month
41,blinkit_718288.0,2026-06-30,22.013367,12.03335,65.027811,61.459161,0.305689,0.167101,0.903010,0.853454,...,0.0,0.0,0.0,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M
42,blinkit_718288.0,2026-07-31,22.013367,12.03335,71.497767,59.294777,0.305689,0.167101,0.992856,0.823398,...,0.0,0.0,0.0,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+1
43,blinkit_718288.0,2026-08-31,22.013367,12.03335,66.543497,63.431688,0.305689,0.167101,0.924058,0.880846,...,0.0,0.0,0.0,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+2
44,blinkit_718288.0,2026-09-30,22.013367,12.03335,63.594309,59.532067,0.305689,0.167101,0.883104,0.826694,...,0.0,0.0,0.0,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3
45,blinkit_718288.0,2026-10-31,22.013367,12.03335,74.309960,64.951819,0.305689,0.167101,1.031907,0.901955,...,0.0,0.0,0.0,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26442,zepto_810439.0,2026-09-30,0.827633,0.89215,0.274929,0.310768,0.026113,0.028149,0.008674,0.009805,...,0.0,0.0,0.0,2026-05-31,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3
26443,zepto_810439.0,2026-10-31,0.827633,0.89215,0.231432,0.302197,0.026113,0.028149,0.007302,0.009535,...,0.0,0.0,0.0,2026-05-31,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+4
26444,zepto_810439.0,2026-11-30,0.827633,0.89215,0.411081,0.354295,0.026113,0.028149,0.012970,0.011178,...,0.0,0.0,0.0,2026-05-31,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+5
26445,zepto_810439.0,2026-12-31,0.827633,0.89215,0.258753,0.331626,0.026113,0.028149,0.008164,0.010463,...,0.0,0.0,0.0,2026-05-31,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+6


In [105]:
trend_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-06-30,2026-05-31


In [106]:
prophet_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-06-30,2026-05-31


In [107]:
trend_file_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7'],
      dtype=object)

In [108]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")

In [109]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [110]:
trend_file_df['portfolio'].isna().sum()

0

In [111]:
prophet_file_df[
    ['month_date', 'key', 'run_month']
].duplicated().sum()

0

In [112]:
prophet_file_df

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,yhat_60_%ile,yhat_70_%ile,yhat_75_%ile,trend_60_%ile,...,vol_in_rum_value,yhat_value,Model_Run,Model_Type,type,train_till,run,step,file_path,run_month
0,2023-01-31,12.156025,7.418186,20.685347,12.156025,12.156025,15.494658,16.919146,17.812126,12.156025,...,0.313613,0.196376,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
1,2023-02-28,13.333243,8.475847,22.383898,13.333243,13.333243,17.084799,18.337736,19.032938,13.333243,...,0.243958,0.218676,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
2,2023-03-31,14.636593,19.947020,33.834941,14.636593,14.636593,28.386028,29.713462,30.485965,14.636593,...,0.351773,0.373116,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
3,2023-04-30,15.897898,3.915806,17.260081,15.897898,15.897898,11.706763,13.267045,13.889709,15.897898,...,0.303948,0.144200,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
4,2023-05-31,17.201247,6.459676,19.736413,17.201247,17.201247,14.121985,15.667556,16.437167,17.201247,...,0.220240,0.180359,Yes,prophet,training,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26442,2026-09-30,0.333859,0.274929,0.274929,0.333859,0.333859,0.274929,0.274929,0.274929,0.333859,...,0.000000,0.008674,Yes,prophet,testing,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
26443,2026-10-31,0.334250,0.231432,0.231432,0.334250,0.334250,0.231432,0.231432,0.231432,0.334250,...,0.000000,0.007302,Yes,prophet,testing,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
26444,2026-11-30,0.334629,0.411081,0.411081,0.334629,0.334629,0.411081,0.411081,0.411081,0.334629,...,0.000000,0.012970,Yes,prophet,testing,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30
26445,2026-12-31,0.335020,0.258753,0.258753,0.335020,0.335020,0.258753,0.258753,0.258753,0.335020,...,0.000000,0.008164,Yes,prophet,testing,2026-05-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-06-30


In [113]:
# Merge 70th percentile Prophet predictions
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    prophet_file_df[['month_date', 'key', 'run_month', 'yhat_70_%ile', 'yhat_60_%ile']].rename(
        columns={
            'yhat_70_%ile': 'pred_prophet_70%ile',
            'yhat_60_%ile': 'pred_prophet_60%ile'
        }
    ),
    on=['month_date', 'key', 'run_month'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [114]:
assert trend_file_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0

In [115]:
trend_file_df.drop('vol_in_rum', axis=1, inplace=True)

In [116]:
offtake_df.head()

,month_date,key,platform_name,parent_material_code,brand_code,vol_in_rum,run_month,imputed
0,2023-01-31,blinkit_718288.0,blinkit,718288,SAFF GOLD,22.584,2026-06-30,0
1,2023-02-28,blinkit_718288.0,blinkit,718288,SAFF GOLD,17.568,2026-06-30,0
2,2023-03-31,blinkit_718288.0,blinkit,718288,SAFF GOLD,25.332,2026-06-30,0
3,2023-04-30,blinkit_718288.0,blinkit,718288,SAFF GOLD,21.888,2026-06-30,0
4,2023-05-31,blinkit_718288.0,blinkit,718288,SAFF GOLD,15.860,2026-06-30,0


In [117]:
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0

In [118]:
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])

In [119]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    offtake_df[['key', 'run_month','month_date', 'vol_in_rum']],
    on=['run_month','month_date', 'key'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [120]:
trend_file_df.select_dtypes('number').isna().sum()

pred_p3m                    0
pred_p6m                    0
pred_prophet                0
pred_rf                     0
pred_value_p3m              0
pred_value_p6m              0
pred_value_prophet          0
pred_value_rf               0
parent_material_code        0
qtr_ind_rate                0
vol_in_rum_value            0
vol_in_rum_treated          0
vol_in_rum_value_treated    0
cov                         0
pred_prophet_70%ile         0
pred_prophet_60%ile         0
vol_in_rum                  0
dtype: int64

In [121]:
trend_file_df[trend_file_df['month_date'] == '2026-07-31']['pred_value_rf'].sum()

28.14937166909646

In [122]:
trend_file_df.select_dtypes('number').min().round()

pred_p3m                         0.0
pred_p6m                         0.0
pred_prophet                     0.0
pred_rf                          0.0
pred_value_p3m                   0.0
pred_value_p6m                   0.0
pred_value_prophet               0.0
pred_value_rf                    0.0
parent_material_code        715101.0
qtr_ind_rate                   100.0
vol_in_rum_value                 0.0
vol_in_rum_treated               0.0
vol_in_rum_value_treated         0.0
cov                              0.0
pred_prophet_70%ile          -1033.0
pred_prophet_60%ile          -1460.0
vol_in_rum                       0.0
dtype: float64

In [123]:
trend_file_df['vol_in_rum'].fillna(0, inplace=True)

In [124]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'parent_material_code', 'platform_name', 'brand_code',
       'qtr_ind_rate', 'vol_in_rum_value', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'run_month', 'M month', 'portfolio', 'pred_prophet_70%ile',
       'pred_prophet_60%ile', 'vol_in_rum'],
      dtype='object')

In [125]:
for col in [ 'pred_prophet_70%ile','pred_prophet_60%ile', 'vol_in_rum']:
    trend_file_df[col] = trend_file_df[col].clip(lower=0)

In [126]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum
0,blinkit_718288.0,2023-01-31,21.828000,21.337833,14.141480,23.283180,0.303115,0.296308,0.196376,0.323322,...,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,16.919146,15.494658,22.584
1,blinkit_718288.0,2023-02-28,21.828000,21.337833,15.747385,21.731667,0.303115,0.296308,0.218676,0.301777,...,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,18.337736,17.084799,17.568
2,blinkit_718288.0,2023-03-31,21.828000,21.337833,26.868887,25.585143,0.303115,0.296308,0.373116,0.355289,...,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,29.713462,28.386028,25.332
3,blinkit_718288.0,2023-04-30,21.828000,21.337833,10.384195,23.536022,0.303115,0.296308,0.144200,0.326834,...,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,13.267045,11.706763,21.888
4,blinkit_718288.0,2023-05-31,21.596000,21.337833,12.988040,21.183205,0.299893,0.296308,0.180359,0.294161,...,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,15.667556,14.121985,15.860
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26442,zepto_810439.0,2026-09-30,0.827633,0.892150,0.274929,0.310768,0.026113,0.028149,0.008674,0.009805,...,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3,Foods,0.274929,0.274929,0.000
26443,zepto_810439.0,2026-10-31,0.827633,0.892150,0.231432,0.302197,0.026113,0.028149,0.007302,0.009535,...,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+4,Foods,0.231432,0.231432,0.000
26444,zepto_810439.0,2026-11-30,0.827633,0.892150,0.411081,0.354295,0.026113,0.028149,0.012970,0.011178,...,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+5,Foods,0.411081,0.411081,0.000
26445,zepto_810439.0,2026-12-31,0.827633,0.892150,0.258753,0.331626,0.026113,0.028149,0.008164,0.010463,...,0.162862,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+6,Foods,0.258753,0.258753,0.000


In [127]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [128]:
trend_file_df['P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

trend_file_df['LY P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['LY P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [129]:
trend_file_df['LY P3M_copy'] = trend_file_df['LY P3M'].copy()

In [130]:
trend_file_df['P3M Max'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()

In [131]:
trend_file_df['P3M Top 2 Mean'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False

In [132]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [133]:
trend_file_df['MoM P3M growth'] = (
    trend_file_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)

In [134]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['MoM P3M growth_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
trend_file_df['MoM P3M growth_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)



In [135]:
trend_file_df['>=20%_3M_inc_month_count'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 

In [136]:
trend_file_df['Avg(P3M Mean, Max)'] = trend_file_df[['P3M', 'P3M Max']].mean(axis=1)

In [137]:
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
#         trend_file_df[col] = trend_file_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )

In [138]:
# trend_file_df.to_csv('collate_check.csv', index=False)

In [139]:
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    trend_file_df[f'{col}_value'] = trend_file_df[col] * trend_file_df['qtr_ind_rate'] / (10 ** 7)

In [140]:
trend_file_df['vol_in_rum_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['vol_in_rum'] / (10 ** 7)
trend_file_df['pred_prophet_70%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_70%ile'] / (10 ** 7)
trend_file_df['pred_prophet_60%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_60%ile'] / (10 ** 7)

In [141]:
value_cols = [col for col in trend_file_df.columns if 'value' in col]
value_cols

['pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'vol_in_rum_value',
 'vol_in_rum_value_treated',
 'P3M_value',
 'P6M_value',
 'LY P3M_value',
 'LY P6M_value',
 'pred_prophet_70%ile_value',
 'pred_prophet_60%ile_value']

In [142]:
for col in value_cols:
    try:
        assert trend_file_df[col].min() >= 0
    except:
        print(col)
    

    # trend_file_df[col] = trend_file_df[col] / (10 ** 7)

In [143]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value
1878,blinkit_718589.0,2023-01-31,140.200000,120.650000,125.240172,124.714531,0.006966,0.005994,6.222288e-03,0.006196,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.006712,6.481297e-03
1879,blinkit_718589.0,2023-02-28,140.200000,120.650000,120.212121,131.282560,0.006966,0.005994,5.972480e-03,0.006522,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.006532,6.251861e-03
1880,blinkit_718589.0,2023-03-31,140.200000,120.650000,127.151290,146.810595,0.006966,0.005994,6.317238e-03,0.007294,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.006815,6.553178e-03
1881,blinkit_718589.0,2023-04-30,140.200000,120.650000,84.135918,121.411359,0.006966,0.005994,4.180112e-03,0.006032,...,NaN,NaN,NaN,140.2,0.006966,NaN,NaN,NaN,0.004649,4.390006e-03
1882,blinkit_718589.0,2023-05-31,131.200000,120.650000,105.836774,104.297415,0.006518,0.005994,5.258272e-03,0.005182,...,NaN,NaN,NaN,131.2,0.006518,NaN,NaN,NaN,0.005686,5.472316e-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12046,swiggy_719193.0,2026-09-30,0.826333,203.230167,0.000000,0.000000,0.000008,0.002032,0.000000e+00,0.000000,...,-100.0,-100.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00
12047,swiggy_719193.0,2026-10-31,0.826333,203.230167,0.064315,0.111000,0.000008,0.002032,6.431450e-07,0.000001,...,-100.0,-100.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000001,8.739289e-07
12048,swiggy_719193.0,2026-11-30,0.826333,203.230167,0.000000,0.000000,0.000008,0.002032,0.000000e+00,0.000000,...,-100.0,-100.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00
12049,swiggy_719193.0,2026-12-31,0.826333,203.230167,0.000000,0.000000,0.000008,0.002032,0.000000e+00,0.000000,...,-100.0,-100.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00


In [144]:
assert trend_file_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0

In [145]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['LY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

trend_file_df['LLY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


trend_file_df['LY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

trend_file_df['LLY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


trend_file_df['OT_Value_in_Cr_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

trend_file_df['OT_Value_in_Cr_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

trend_file_df['OT_Value_in_Cr_lag_3'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)

In [146]:
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [147]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'parent_material_code', 'platform_name', 'brand_code',
       'qtr_ind_rate', 'vol_in_rum_value', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'run_month', 'M month', 'portfolio', 'pred_prophet_70%ile',
       'pred_prophet_60%ile', 'vol_in_rum', 'P3M', 'P6M', 'LY P3M', 'LY P6M',
       'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'pred_prophet_60%ile_value', 'LY', 'LLY',
       'LY value', 'LLY value', 'OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2',
       'OT_Value_in_Cr_lag_3'],
      dtype='object')

In [148]:
# trend_file_df[['ASM', 'Depot', 'PSKU']] = trend_file_df['key'].str.split('_', expand=True)

In [149]:
trend_file_df.reset_index(drop=True, inplace=True)

In [150]:
trend_file_df.shape

(26447, 53)

In [151]:
trend_file_df['key'].nunique()

686

In [152]:
# batch_info = pd.read_excel(
#     '/data/aniket/az_demand_forecasting-mil-sc/channel_wise_batch.xlsx'
# )

In [153]:
brand_class = pd.read_excel('/data/aman_singh/acuuracy_check/brand_class_new.xlsx')
brand_class['Channel'] = brand_class['Channel'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
brand_class.columns = brand_class.columns.str.lower()
brand_class = brand_class[brand_class['channel'] == 'QCOM']
brand_class = brand_class[['brand','final class']]

brand_class.rename(columns = {'brand':'brand_code','final class':'class'}, inplace = True)


In [154]:
# trend_file_df.drop(columns = ['final class'], inplace = True)

In [155]:
brand_class

,brand_code,class
845,SAFF GOLD,A
846,PCNO(R),A
847,SAFF ACTV,A
848,SFOATS-FL,A
849,SAFF OATS,A
...,...,...
991,PADV-HRAD,C
992,PADVJAS-F,C
993,SFFT_VNGR,C
994,PURSNS_GM,C


In [156]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_class, 
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)
del len_before_merge

In [157]:
trend_file_df['class'].isna().sum()

335

In [158]:
trend_file_df['class'].unique()

array(['C', 'NPD', nan, 'A', 'B'], dtype=object)

In [159]:
# trend_file_df[
#     # (trend_file_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (trend_file_df['month_date'] > '2024-06-30') &
#     (trend_file_df['M month'].notna())
#     # (trend_file_df['class'].isin(['B', 'C']))
# ].to_csv('Heuristic_QCOM_Chain_PSKU_Offtakes_live2.csv', index=False)

missing combinations

In [160]:
model_file = trend_file_df.copy()

In [161]:
model_file['key'].nunique()

686

In [162]:
run_month

Timestamp('2026-06-30 00:00:00')

In [163]:
data_query = f"""select * from {input_table} where run_month = '{run_month_inp}' and month_date >= '2023-01-31' and platform_name in ('blinkit', 'swiggy', 'zepto')"""
qcom_df = pd.read_sql(data_query, dev_conn)
qcom_df.head()

,MONTH_DATE,KEY,PLATFORM_NAME,PARENT_MATERIAL_CODE,BRAND_CODE,VOL_IN_RUM,RUN_MONTH,IMPUTED
0,2023-01-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,22.584,2026-06-30,0
1,2023-02-28,blinkit_718288,blinkit,718288.0,SAFF GOLD,17.568,2026-06-30,0
2,2023-03-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,25.332,2026-06-30,0
3,2023-04-30,blinkit_718288,blinkit,718288.0,SAFF GOLD,21.888,2026-06-30,0
4,2023-05-31,blinkit_718288,blinkit,718288.0,SAFF GOLD,15.860,2026-06-30,0


In [164]:
qcom_df.columns = qcom_df.columns.str.lower()

In [165]:
qcom_df['key'] = qcom_df[['platform_name', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [166]:
qcom_df

,month_date,key,platform_name,parent_material_code,brand_code,vol_in_rum,run_month,imputed
0,2023-01-31,blinkit_718288.0,blinkit,718288.0,SAFF GOLD,22.584,2026-06-30,0
1,2023-02-28,blinkit_718288.0,blinkit,718288.0,SAFF GOLD,17.568,2026-06-30,0
2,2023-03-31,blinkit_718288.0,blinkit,718288.0,SAFF GOLD,25.332,2026-06-30,0
3,2023-04-30,blinkit_718288.0,blinkit,718288.0,SAFF GOLD,21.888,2026-06-30,0
4,2023-05-31,blinkit_718288.0,blinkit,718288.0,SAFF GOLD,15.860,2026-06-30,0
...,...,...,...,...,...,...,...,...
31829,2027-03-31,zepto_810685.0,zepto,810685.0,SAF-MUSLI,0.000,2026-06-30,0
31830,2027-03-31,zepto_810738.0,zepto,810738.0,PABABY_GM,0.000,2026-06-30,0
31831,2027-03-31,zepto_810971.0,zepto,810971.0,PA_ESS_HO,0.000,2026-06-30,0
31832,2027-03-31,zepto_811005.0,zepto,811005.0,PA_ESS_HO,0.000,2026-06-30,0


In [167]:
model_file['run_month'] = pd.to_datetime(model_file['run_month'])
model_file['month_date'] = pd.to_datetime(model_file['month_date'])

qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month_date'] = pd.to_datetime(qcom_df['month_date'])

In [168]:
tmp_df = model_file.groupby(['key', 'run_month'])['LY'].count().reset_index()
tmp_df#.isnull().sum()
#qcom_df[~qcom_df['key'].isin(model_file['key'].unique())]
qcom_df = qcom_df.merge(tmp_df, on = ['key', 'run_month'], how = 'left')
missing_df = qcom_df[qcom_df['LY'].isna()]
missing_df


,month_date,key,platform_name,parent_material_code,brand_code,vol_in_rum,run_month,imputed,LY
985,2023-01-31,blinkit_718465.0,blinkit,718465.0,SAFF OATS,6.2552,2026-06-30,0,NaN
986,2023-02-28,blinkit_718465.0,blinkit,718465.0,SAFF OATS,5.8856,2026-06-30,0,NaN
987,2023-03-31,blinkit_718465.0,blinkit,718465.0,SAFF OATS,5.6896,2026-06-30,0,NaN
988,2023-04-30,blinkit_718465.0,blinkit,718465.0,SAFF OATS,6.2720,2026-06-30,0,NaN
989,2023-05-31,blinkit_718465.0,blinkit,718465.0,SAFF OATS,7.2306,2026-06-30,0,NaN
...,...,...,...,...,...,...,...,...,...
31829,2027-03-31,zepto_810685.0,zepto,810685.0,SAF-MUSLI,0.0000,2026-06-30,0,NaN
31830,2027-03-31,zepto_810738.0,zepto,810738.0,PABABY_GM,0.0000,2026-06-30,0,NaN
31831,2027-03-31,zepto_810971.0,zepto,810971.0,PA_ESS_HO,0.0000,2026-06-30,0,NaN
31832,2027-03-31,zepto_811005.0,zepto,811005.0,PA_ESS_HO,0.0000,2026-06-30,0,NaN


In [169]:
missing_df['key'].nunique()

217

In [170]:

missing_df = missing_df[['key','run_month','month_date', 'platform_name', 'parent_material_code', 'brand_code',
       'vol_in_rum']]
missing_df

,key,run_month,month_date,platform_name,parent_material_code,brand_code,vol_in_rum
985,blinkit_718465.0,2026-06-30,2023-01-31,blinkit,718465.0,SAFF OATS,6.2552
986,blinkit_718465.0,2026-06-30,2023-02-28,blinkit,718465.0,SAFF OATS,5.8856
987,blinkit_718465.0,2026-06-30,2023-03-31,blinkit,718465.0,SAFF OATS,5.6896
988,blinkit_718465.0,2026-06-30,2023-04-30,blinkit,718465.0,SAFF OATS,6.2720
989,blinkit_718465.0,2026-06-30,2023-05-31,blinkit,718465.0,SAFF OATS,7.2306
...,...,...,...,...,...,...,...
31829,zepto_810685.0,2026-06-30,2027-03-31,zepto,810685.0,SAF-MUSLI,0.0000
31830,zepto_810738.0,2026-06-30,2027-03-31,zepto,810738.0,PABABY_GM,0.0000
31831,zepto_810971.0,2026-06-30,2027-03-31,zepto,810971.0,PA_ESS_HO,0.0000
31832,zepto_811005.0,2026-06-30,2027-03-31,zepto,811005.0,PA_ESS_HO,0.0000


In [171]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()




Credentials retrieved successfully for prod db.


,month_date,brand_code,qtr_ind_rate
0,2027-03-31,TRU_PDRFR,850.57000
1,2027-03-31,TRU_OATS,177.07000
2,2027-03-31,TRU_QUINO,204.75000
3,2027-03-31,TRU_RAW,453.44000
4,2027-03-31,NHR_NHO_E,266.57953


In [172]:
len_before_merge = len(missing_df)

missing_df = missing_df.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(missing_df)

In [173]:
missing_df

,key,run_month,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,qtr_ind_rate
0,blinkit_718465.0,2026-06-30,2023-01-31,blinkit,718465.0,SAFF OATS,6.2552,127515.619406
1,blinkit_718465.0,2026-06-30,2023-02-28,blinkit,718465.0,SAFF OATS,5.8856,127515.619406
2,blinkit_718465.0,2026-06-30,2023-03-31,blinkit,718465.0,SAFF OATS,5.6896,127515.619406
3,blinkit_718465.0,2026-06-30,2023-04-30,blinkit,718465.0,SAFF OATS,6.2720,127515.619406
4,blinkit_718465.0,2026-06-30,2023-05-31,blinkit,718465.0,SAFF OATS,7.2306,127515.619406
...,...,...,...,...,...,...,...,...
4010,zepto_810685.0,2026-06-30,2027-03-31,zepto,810685.0,SAF-MUSLI,0.0000,315513.490535
4011,zepto_810738.0,2026-06-30,2027-03-31,zepto,810738.0,PABABY_GM,0.0000,366.484998
4012,zepto_810971.0,2026-06-30,2027-03-31,zepto,810971.0,PA_ESS_HO,0.0000,12860.631072
4013,zepto_811005.0,2026-06-30,2027-03-31,zepto,811005.0,PA_ESS_HO,0.0000,12860.631072


In [174]:
missing_df['month_date'] = pd.to_datetime(missing_df['month_date'])
missing_df['run_month'] = pd.to_datetime(missing_df['run_month'])


mappings = {}

for run_month in missing_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   
missing_df['M month'] = missing_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(missing_df)

# Merge 70th percentile Prophet predictions

assert missing_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0
missing_df.drop('vol_in_rum', axis=1, inplace=True)




In [175]:
offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    offtake_df[['key', 'month_date', 'vol_in_rum']],
    on=['month_date', 'key'],
    how='left'
)
assert len(missing_df) == len_before_merge

missing_df['vol_in_rum'].fillna(0, inplace=True)
for col in [ 'vol_in_rum']:
    missing_df[col] = missing_df[col].clip(lower=0)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=1).mean()

missing_df['P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

missing_df['LY P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

missing_df['LY P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()
missing_df['LY P3M_copy'] = missing_df['LY P3M'].copy()
missing_df['P3M Max'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()
missing_df['P3M Top 2 Mean'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth'] = (
    missing_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
missing_df['MoM P3M growth_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)


missing_df['>=20%_3M_inc_month_count'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 
missing_df['Avg(P3M Mean, Max)'] = missing_df[['P3M', 'P3M Max']].mean(axis=1)
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
#         missing_df[col] = missing_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )
# missing_df.to_csv('collate_check.csv', index=False)
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    missing_df[f'{col}_value'] = missing_df[col] * missing_df['qtr_ind_rate'] / (10 ** 7)
missing_df['vol_in_rum_value'] = missing_df['qtr_ind_rate'] * missing_df['vol_in_rum'] / (10 ** 7)
value_cols = [col for col in missing_df.columns if 'value' in col]
value_cols
for col in value_cols:
    try:
        assert missing_df[col].min() >= 0
    except:
        print(col)
    

    # missing_df[col] = missing_df[col] / (10 ** 7)
missing_df
assert missing_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['LY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

missing_df['LLY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


missing_df['LY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

missing_df['LLY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


missing_df['OT_Value_in_Cr_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

missing_df['OT_Value_in_Cr_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

missing_df['OT_Value_in_Cr_lag_3'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

missing_df.reset_index(drop=True, inplace=True)

# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()
# brand_class_df.columns = ['brand_code', 'class']

len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    brand_class, 
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(missing_df)
del len_before_merge
missing_df['class'].isna().sum()
missing_df['class'].unique()

array(['C', 'NPD', 'A', 'B'], dtype=object)

In [176]:
pd.set_option('display.max_columns', None)

In [177]:
# missing_df[
#     # (missing_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (missing_df['month_date'] > '2024-06-30') &
#     (missing_df['M month'].notna())
#     # (missing_df['class'].isin(['B', 'C']))
# ].to_csv('missing_combinations_QCOM_Chain_city_PSKU_Offtakes.csv', index=False)

In [178]:
model_file

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class
0,blinkit_718589.0,2023-01-31,140.200000,120.650000,125.240172,124.714531,0.006966,0.005994,6.222288e-03,0.006196,718589.0,blinkit,ADV-AHO-R,496.828458,0.006126,123.3,0.006126,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,135.101501,130.453417,123.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.006712,6.481297e-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C
1,blinkit_718589.0,2023-02-28,140.200000,120.650000,120.212121,131.282560,0.006966,0.005994,5.972480e-03,0.006522,718589.0,blinkit,ADV-AHO-R,496.828458,0.007214,145.2,0.007214,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,131.464418,125.835409,145.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.006532,6.251861e-03,NaN,NaN,NaN,NaN,0.006126,NaN,NaN,C
2,blinkit_718589.0,2023-03-31,140.200000,120.650000,127.151290,146.810595,0.006966,0.005994,6.317238e-03,0.007294,718589.0,blinkit,ADV-AHO-R,496.828458,0.007557,152.1,0.007557,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,137.178128,131.900206,152.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.006815,6.553178e-03,NaN,NaN,NaN,NaN,0.007214,0.006126,NaN,C
3,blinkit_718589.0,2023-04-30,140.200000,120.650000,84.135918,121.411359,0.006966,0.005994,4.180112e-03,0.006032,718589.0,blinkit,ADV-AHO-R,496.828458,0.004784,96.3,0.004784,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,93.565810,88.360591,96.3,140.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,140.2,0.006966,NaN,NaN,NaN,0.004649,4.390006e-03,NaN,NaN,NaN,NaN,0.007557,0.007214,0.006126,C
4,blinkit_718589.0,2023-05-31,131.200000,120.650000,105.836774,104.297415,0.006518,0.005994,5.258272e-03,0.005182,718589.0,blinkit,ADV-AHO-R,496.828458,0.004650,93.6,0.004650,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,114.438626,110.144987,93.6,131.2,NaN,NaN,NaN,NaN,NaN,NaN,-6.419401,NaN,NaN,NaN,131.2,0.006518,NaN,NaN,NaN,0.005686,5.472316e-03,NaN,NaN,NaN,NaN,0.004784,0.007557,0.007214,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26442,swiggy_719193.0,2026-09-30,0.826333,203.230167,0.000000,0.000000,0.000008,0.002032,0.000000e+00,0.000000,719193.0,swiggy,VEG_CLEAN,100.000000,0.000000,0.0,0.000000,2026-05-31,4.357389,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3,Health & Hygiene,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-100.000000,-100.0,-100.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,NaN
26443,swiggy_719193.0,2026-10-31,0.826333,203.230167,0.064315,0.111000,0.000008,0.002032,6.431450e-07,0.000001,719193.0,swiggy,VEG_CLEAN,100.000000,0.000000,0.0,0.000000,2026-05-31,4.357389,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+4,Health & Hygiene,0.102584,0.087393,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-100.000000,-100.0,-100.0,0.0,0.0

In [179]:
model_file['skipped'] = 0
missing_df['skipped'] = 1
final_df = pd.concat([model_file,missing_df])
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped
0,blinkit_718589.0,2023-01-31,140.2,120.65,125.240172,124.714531,0.006966,0.005994,0.006222,0.006196,718589.0,blinkit,ADV-AHO-R,496.828458,0.006126,123.3,0.006126,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,135.101501,130.453417,123.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.006712,0.006481,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C,0
1,blinkit_718589.0,2023-02-28,140.2,120.65,120.212121,131.282560,0.006966,0.005994,0.005972,0.006522,718589.0,blinkit,ADV-AHO-R,496.828458,0.007214,145.2,0.007214,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,131.464418,125.835409,145.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.006532,0.006252,NaN,NaN,NaN,NaN,0.006126,NaN,NaN,C,0
2,blinkit_718589.0,2023-03-31,140.2,120.65,127.151290,146.810595,0.006966,0.005994,0.006317,0.007294,718589.0,blinkit,ADV-AHO-R,496.828458,0.007557,152.1,0.007557,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,137.178128,131.900206,152.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.006815,0.006553,NaN,NaN,NaN,NaN,0.007214,0.006126,NaN,C,0
3,blinkit_718589.0,2023-04-30,140.2,120.65,84.135918,121.411359,0.006966,0.005994,0.004180,0.006032,718589.0,blinkit,ADV-AHO-R,496.828458,0.004784,96.3,0.004784,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,93.565810,88.360591,96.3,140.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,140.200000,0.006966,NaN,NaN,NaN,0.004649,0.004390,NaN,NaN,NaN,NaN,0.007557,0.007214,0.006126,C,0
4,blinkit_718589.0,2023-05-31,131.2,120.65,105.836774,104.297415,0.006518,0.005994,0.005258,0.005182,718589.0,blinkit,ADV-AHO-R,496.828458,0.004650,93.6,0.004650,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,114.438626,110.144987,93.6,131.2,NaN,NaN,NaN,NaN,NaN,NaN,-6.419401,NaN,NaN,NaN,131.200000,0.006518,NaN,NaN,NaN,0.005686,0.005472,NaN,NaN,NaN,NaN,0.004784,0.007557,0.007214,C,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4010,zepto_810125.0,2026-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,810125.0,zepto,SW_SGPRF,1712.605337,0.000000,NaN,NaN,NaT,NaN,NaN,NaN,NaN,2026-06-30,M+5,Male Grooming,NaN,NaN,0.0,3.6,NaN,NaN,NaN,NaN,1.493333,1.046667,141.071429,148.888889,114.285714,3.0,2.546667,0.000617,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001178,0.000562,0.000110,NPD,1
4011,zepto_810125.0,2026-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,810125.0,zepto,SW_SGPRF,1712.605337,0.000000,NaN,NaN,NaT,NaN,NaN,NaN,NaN,2026-06-30,M+6,Male Grooming,NaN,NaN,0.0,3.6,NaN,NaN,NaN,NaN,1.493333,1.046667,141.071429,148.888889,114.285714,3.0,2.546667,0.000617,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001178,0.000562,0.000110,NPD,1
4012,zepto_810125.0,2027-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,810125.0,zepto,SW_SGPRF,1712.605337,0.000000,NaN,NaN,NaT,NaN,NaN,NaN,NaN,2026-06-30,M+7,Male Grooming,NaN,NaN,0.0,3.6,NaN,NaN,NaN,NaN,1.493333,1.046667

In [180]:
final_df[final_df.select_dtypes(include='number').columns] = final_df.select_dtypes(include='number').fillna(0)
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped
0,blinkit_718589.0,2023-01-31,140.2,120.65,125.240172,124.714531,0.006966,0.005994,0.006222,0.006196,718589.0,blinkit,ADV-AHO-R,496.828458,0.006126,123.3,0.006126,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,135.101501,130.453417,123.3,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.006712,0.006481,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,C,0
1,blinkit_718589.0,2023-02-28,140.2,120.65,120.212121,131.282560,0.006966,0.005994,0.005972,0.006522,718589.0,blinkit,ADV-AHO-R,496.828458,0.007214,145.2,0.007214,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,131.464418,125.835409,145.2,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.006532,0.006252,0.00,0.0,0.000000,0.0,0.006126,0.000000,0.000000,C,0
2,blinkit_718589.0,2023-03-31,140.2,120.65,127.151290,146.810595,0.006966,0.005994,0.006317,0.007294,718589.0,blinkit,ADV-AHO-R,496.828458,0.007557,152.1,0.007557,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,137.178128,131.900206,152.1,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.006815,0.006553,0.00,0.0,0.000000,0.0,0.007214,0.006126,0.000000,C,0
3,blinkit_718589.0,2023-04-30,140.2,120.65,84.135918,121.411359,0.006966,0.005994,0.004180,0.006032,718589.0,blinkit,ADV-AHO-R,496.828458,0.004784,96.3,0.004784,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,93.565810,88.360591,96.3,140.2,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,140.200000,0.006966,0.0,0.0,0.0,0.004649,0.004390,0.00,0.0,0.000000,0.0,0.007557,0.007214,0.006126,C,0
4,blinkit_718589.0,2023-05-31,131.2,120.65,105.836774,104.297415,0.006518,0.005994,0.005258,0.005182,718589.0,blinkit,ADV-AHO-R,496.828458,0.004650,93.6,0.004650,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,114.438626,110.144987,93.6,131.2,0.0,0.0,0.0,0.0,0.000000,0.000000,-6.419401,0.000000,0.000000,0.0,131.200000,0.006518,0.0,0.0,0.0,0.005686,0.005472,0.00,0.0,0.000000,0.0,0.004784,0.007557,0.007214,C,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4010,zepto_810125.0,2026-11-30,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125.0,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-06-30,M+5,Male Grooming,0.000000,0.000000,0.0,3.6,0.0,0.0,0.0,0.0,1.493333,1.046667,141.071429,148.888889,114.285714,3.0,2.546667,0.000617,0.0,0.0,0.0,0.000000,0.000000,0.00,0.0,0.000000,0.0,0.001178,0.000562,0.000110,NPD,1
4011,zepto_810125.0,2026-12-31,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125.0,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-06-30,M+6,Male Grooming,0.000000,0.000000,0.0,3.6,0.0,0.0,0.0,0.0,1.493333,1.046667,141.071429

In [181]:
final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped
0,blinkit_718589.0,2023-01-31,140.2,120.65,125.240172,124.714531,0.006966,0.005994,0.006222,0.006196,718589.0,blinkit,ADV-AHO-R,496.828458,0.006126,123.3,0.006126,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,135.101501,130.453417,123.3,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.006712,0.006481,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,C,0
1,blinkit_718589.0,2023-02-28,140.2,120.65,120.212121,131.282560,0.006966,0.005994,0.005972,0.006522,718589.0,blinkit,ADV-AHO-R,496.828458,0.007214,145.2,0.007214,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,131.464418,125.835409,145.2,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.006532,0.006252,0.00,0.0,0.000000,0.0,0.006126,0.000000,0.000000,C,0
2,blinkit_718589.0,2023-03-31,140.2,120.65,127.151290,146.810595,0.006966,0.005994,0.006317,0.007294,718589.0,blinkit,ADV-AHO-R,496.828458,0.007557,152.1,0.007557,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,137.178128,131.900206,152.1,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.006815,0.006553,0.00,0.0,0.000000,0.0,0.007214,0.006126,0.000000,C,0
3,blinkit_718589.0,2023-04-30,140.2,120.65,84.135918,121.411359,0.006966,0.005994,0.004180,0.006032,718589.0,blinkit,ADV-AHO-R,496.828458,0.004784,96.3,0.004784,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,93.565810,88.360591,96.3,140.2,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,140.200000,0.006966,0.0,0.0,0.0,0.004649,0.004390,0.00,0.0,0.000000,0.0,0.007557,0.007214,0.006126,C,0
4,blinkit_718589.0,2023-05-31,131.2,120.65,105.836774,104.297415,0.006518,0.005994,0.005258,0.005182,718589.0,blinkit,ADV-AHO-R,496.828458,0.004650,93.6,0.004650,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,114.438626,110.144987,93.6,131.2,0.0,0.0,0.0,0.0,0.000000,0.000000,-6.419401,0.000000,0.000000,0.0,131.200000,0.006518,0.0,0.0,0.0,0.005686,0.005472,0.00,0.0,0.000000,0.0,0.004784,0.007557,0.007214,C,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4010,zepto_810125.0,2026-11-30,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125.0,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-06-30,M+5,Male Grooming,0.000000,0.000000,0.0,3.6,0.0,0.0,0.0,0.0,1.493333,1.046667,141.071429,148.888889,114.285714,3.0,2.546667,0.000617,0.0,0.0,0.0,0.000000,0.000000,0.00,0.0,0.000000,0.0,0.001178,0.000562,0.000110,NPD,1
4011,zepto_810125.0,2026-12-31,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125.0,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-06-30,M+6,Male Grooming,0.000000,0.000000,0.0,3.6,0.0,0.0,0.0,0.0,1.493333,1.046667,141.071429

In [182]:
# final_df[
#     # (final_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (final_df['month_date'] > '2024-06-30') &
#     (final_df['M month'].notna())
#     # (missing_df['class'].isin(['B', 'C']))
# ].to_csv('all_combinations_QCOM_chain_PSKU_Offtakes_DEC2.csv', index=False)

In [183]:
# final_df.to_csv('heuristic_data_preprocessed_qcom_jan.csv')

In [184]:
# import pandas as pd
# final_df = pd.read_csv('/data/aman_singh/acuuracy_check/heuristic_data_preprocessed_qcom_cp_live_jan.csv')
# final_df

In [185]:
final_df[(final_df['M month'].notna())]#['key'].nunique()

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped
41,blinkit_718589.0,2026-06-30,143.007831,1583.503916,430.049613,357.918778,0.007105,0.078673,0.021366,0.017782,718589.0,blinkit,ADV-AHO-R,496.828458,0.0,0.0,0.0,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M,Hair Oils,439.778060,434.523044,0.0,376.7,373.25,210.6,200.6,210.6,371.500000,370.650000,3.888582,-2.395693,0.459708,0.0,374.100000,0.018716,0.018544,0.010463,0.009966,0.021849,0.021588,258.30,109.2,0.012833,0.005425,0.020733,0.016619,0.018795,C,0
42,blinkit_718589.0,2026-07-31,143.007831,1583.503916,450.795742,366.006873,0.007105,0.078673,0.022397,0.018184,718589.0,blinkit,ADV-AHO-R,496.828458,0.0,0.0,0.0,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+1,Hair Oils,460.814726,455.897267,0.0,376.7,373.25,210.6,208.4,227.5,371.500000,370.650000,3.888582,-2.395693,0.459708,0.0,374.100000,0.018716,0.018544,0.010463,0.010354,0.022895,0.022650,310.80,124.8,0.015441,0.006200,0.020733,0.016619,0.018795,C,0
43,blinkit_718589.0,2026-08-31,143.007831,1583.503916,467.268542,358.954944,0.007105,0.078673,0.023215,0.017834,718589.0,blinkit,ADV-AHO-R,496.828458,0.0,0.0,0.0,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+2,Hair Oils,476.953543,471.691663,0.0,376.7,373.25,210.6,226.5,267.3,371.500000,370.650000,3.888582,-2.395693,0.459708,0.0,374.100000,0.018716,0.018544,0.010463,0.011253,0.023696,0.023435,317.10,112.2,0.015754,0.005574,0.020733,0.016619,0.018795,C,0
44,blinkit_718589.0,2026-09-30,143.007831,1583.503916,472.422296,369.400278,0.007105,0.078673,0.023471,0.018353,718589.0,blinkit,ADV-AHO-R,496.828458,0.0,0.0,0.0,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3,Hair Oils,484.688850,479.034666,0.0,376.7,373.25,210.6,253.0,295.4,371.500000,370.650000,3.888582,-2.395693,0.459708,0.0,374.100000,0.018716,0.018544,0.010463,0.012570,0.024081,0.023800,305.40,118.2,0.015173,0.005873,0.020733,0.016619,0.018795,C,0
45,blinkit_718589.0,2026-10-31,143.007831,1583.503916,518.026130,371.324706,0.007105,0.078673,0.025737,0.018448,718589.0,blinkit,ADV-AHO-R,496.828458,0.0,0.0,0.0,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+4,Hair Oils,529.716581,524.476941,0.0,376.7,373.25,210.6,269.3,311.1,371.500000,370.650000,3.888582,-2.395693,0.459708,0.0,374.100000,0.018716,0.018544,0.010463,0.013380,0.026318,0.026058,395.10,134.7,0.019630,0.006692,0.020733,0.016619,0.018795,C,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4009,zepto_810125.0,2026-10-31,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125.0,zepto,SW_SGPRF,1712.605337,0.0,0.0,0.0,NaT,0.000000,NaN,NaN,NaN,2026-06-30,M+4,Male Grooming,0.000000,0.000000,0.0,3.6,0.00,0.0,0.0,0.0,1.493333,1.046667,141.071429,148.888889,114.285714,3.0,2.546667,0.000617,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.0,0.000000,0.000000,0.001178,0.000562,0.000110,NPD,1
4010,zepto_810125.0,2026-11-30,0.000000,0.000000,0.000000,0.000000,0.0

Heuristic new approac

In [186]:
# pip install pymannkendall

In [187]:
# final_df.to_csv('brek2.csv', index=False)

In [188]:
import pandas as pd
import numpy as np
import pymannkendall as mk

def detect_trend_for_group(df_grp):
    """
    Detect final trend flag and p3m_slope_flag separately.
    Must contain 'month_date', 'vol_in_rum', 'run_month'
    """

    # ---------- 1. Sort ----------
    df_grp = df_grp.sort_values("month_date")

    # ---------- 2. Identify run_month ----------
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]

    # if no actual data → no trend
    if df_actual.empty or len(df_actual) < 4:
        return pd.Series({"trend_flag": 0, "p3m_slope_flag": 0})

    # ---------- 3. MK Trend ----------
    series = df_actual["vol_in_rum_value"].astype(float)

    try:
        mk_result = mk.original_test(series)
        if mk_result.trend == "increasing":
            mk_trend = 1
        elif mk_result.trend == "decreasing":
            mk_trend = -1
        else:
            mk_trend = 0
    except:
        mk_trend = 0

    # ---------- 4. P3M Slope ----------
    # last 4 months → take last 3 with shift
    #shifted_series = series.shift(1).dropna()

    p3m_values = series.tail(3).values
    #print(p3m_values)

    if len(p3m_values) < 3:
        slope_flag = 0
    else:
        x = np.arange(3)
        slope = np.polyfit(x, p3m_values, 1)[0]
        slope_flag = 1 if slope > 0 else (-1 if slope < 0 else 0)
        

    return pd.Series({
        "trend_flag": mk_trend,
        "p3m_slope_flag": slope_flag
    })


# ---------------------------------------------------------
# APPLY ON ENTIRE DATASET
# ---------------------------------------------------------

# trend_df = final_df.groupby(
#     ["platform_name", "parent_material_code"]
# ).apply(detect_trend_for_group).reset_index()

# trend_df = final_df[final_df['key'] == 'Zepto_721898'].groupby(
#     ["platform_name", "parent_material_code", "run_month"]
# ).apply(detect_trend_for_group).reset_index()
trend_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(detect_trend_for_group).reset_index()

In [189]:
trend_df["final_trend"] = np.where(
    (trend_df["trend_flag"] == 1) & (trend_df["p3m_slope_flag"] == 1), 1,
    np.where(
        (trend_df["trend_flag"] == -1) & (trend_df["p3m_slope_flag"] == -1), -1,
        0
    )
)
trend_df

,platform_name,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend
0,blinkit,718288.0,2026-06-30,1,-1,0
1,blinkit,718310.0,2026-06-30,-1,1,0
2,blinkit,718312.0,2026-06-30,1,1,1
3,blinkit,718315.0,2026-06-30,-1,0,0
4,blinkit,718317.0,2026-06-30,-1,0,0
...,...,...,...,...,...,...
898,zepto,810685.0,2026-06-30,0,1,0
899,zepto,810738.0,2026-06-30,1,-1,0
900,zepto,810971.0,2026-06-30,0,0,0
901,zepto,811005.0,2026-06-30,0,0,0


In [190]:
trend_df[trend_df['final_trend'] == 1]

,platform_name,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend
2,blinkit,718312.0,2026-06-30,1,1,1
7,blinkit,718322.0,2026-06-30,1,1,1
15,blinkit,718371.0,2026-06-30,1,1,1
18,blinkit,718398.0,2026-06-30,1,1,1
21,blinkit,718434.0,2026-06-30,1,1,1
...,...,...,...,...,...,...
797,zepto,731090.0,2026-06-30,1,1,1
826,zepto,808271.0,2026-06-30,1,1,1
860,zepto,809949.0,2026-06-30,1,1,1
863,zepto,810009.0,2026-06-30,1,1,1


## detect seasonality

In [191]:
from statsmodels.tsa.stattools import acf
import numpy as np
import pandas as pd

def detect_yearly_seasonality(df_grp, threshold=0.3):
    """
    Detects yearly seasonality using ACF at lag=12 only.
    Uses vol_in_rum as the metric.
    """
    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]
    series = df_actual["vol_in_rum"].astype(float).values

    # Need at least 18 points to compare last year vs this year
    if len(series) < 18:
        return 0

    # Compute ACF up to lag-12
    acf_vals = acf(series, nlags=12, fft=False)

    lag12_acf = acf_vals[12]

    # absolute ACF because seasonal correlation can be negative as well
    if abs(lag12_acf) >= threshold:
        return 1
    else:
        return 0
    

seasonality_df = final_df.groupby(
    ["platform_name", "brand_code", 'run_month']
).apply(detect_yearly_seasonality).reset_index(name="seasonality_flag")

seasonality_df



,platform_name,brand_code,run_month,seasonality_flag
0,blinkit,ADV-AHO-R,2026-06-30,0
1,blinkit,BIO OILS,2026-06-30,1
2,blinkit,CO_SO_PCP,2026-06-30,0
3,blinkit,CO_SO_VCN,2026-06-30,0
4,blinkit,H&C,2026-06-30,0
...,...,...,...,...
237,zepto,SW HRGEL,2026-06-30,0
238,zepto,SW HSPRY,2026-06-30,0
239,zepto,SW STLDEO,2026-06-30,0
240,zepto,SW_HR_WAX,2026-06-30,0


In [192]:
seasonality_df[seasonality_df['seasonality_flag'] == 1]

,platform_name,brand_code,run_month,seasonality_flag
1,blinkit,BIO OILS,2026-06-30,1
26,blinkit,PADV-HRCR,2026-06-30,1
42,blinkit,P_EN_BGHB,2026-06-30,1
44,blinkit,P_EN_RSMR,2026-06-30,1
46,blinkit,REV.LQDST,2026-06-30,1
50,blinkit,SAFF ACTV,2026-06-30,1
64,blinkit,SFOAT-CUP,2026-06-30,1
66,blinkit,SFOATS_GD,2026-06-30,1
70,blinkit,SF_MNMKHN,2026-06-30,1
73,blinkit,SW HRGEL,2026-06-30,1


In [193]:
# seasonality_df.to_csv('seasonality_qcom.csv')

In [194]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["vol_in_rum_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,platform_name,parent_material_code,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,blinkit,718288.0,2026-06-30,0.617524,1.138415,0.825880,0.104178
1,blinkit,718310.0,2026-06-30,0.000000,0.030547,0.006010,0.008179
2,blinkit,718312.0,2026-06-30,0.081367,0.506730,0.251512,0.085073
3,blinkit,718315.0,2026-06-30,0.000000,0.000000,0.000000,0.000000
4,blinkit,718317.0,2026-06-30,0.000000,0.000000,0.000000,0.000000


In [195]:
trend_df = trend_df.merge(threshold_df, on = ['platform_name', 'parent_material_code', 'run_month'], how = 'left')
trend_df

,platform_name,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend,lower_threshold,upper_threshold,mean_value,std_value
0,blinkit,718288.0,2026-06-30,1,-1,0,0.617524,1.138415,0.825880,0.104178
1,blinkit,718310.0,2026-06-30,-1,1,0,0.000000,0.030547,0.006010,0.008179
2,blinkit,718312.0,2026-06-30,1,1,1,0.081367,0.506730,0.251512,0.085073
3,blinkit,718315.0,2026-06-30,-1,0,0,0.000000,0.000000,0.000000,0.000000
4,blinkit,718317.0,2026-06-30,-1,0,0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
898,zepto,810685.0,2026-06-30,0,1,0,0.000000,0.001548,0.000432,0.000372
899,zepto,810738.0,2026-06-30,1,-1,0,0.000000,0.009291,0.002904,0.002129
900,zepto,810971.0,2026-06-30,0,0,0,0.000386,0.001543,0.000772,0.000238
901,zepto,811005.0,2026-06-30,0,0,0,0.000363,0.001452,0.000726,0.000234


In [196]:
trend_df.to_csv('t_thres_df_qcom.csv')

In [197]:
final_df = final_df.merge(seasonality_df, on = ["platform_name", "brand_code", 'run_month'], how = 'left')
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped,seasonality_flag
0,blinkit_718589.0,2023-01-31,140.2,120.65,125.240172,124.714531,0.006966,0.005994,0.006222,0.006196,718589.0,blinkit,ADV-AHO-R,496.828458,0.006126,123.3,0.006126,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,135.101501,130.453417,123.3,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.006712,0.006481,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,C,0,0
1,blinkit_718589.0,2023-02-28,140.2,120.65,120.212121,131.282560,0.006966,0.005994,0.005972,0.006522,718589.0,blinkit,ADV-AHO-R,496.828458,0.007214,145.2,0.007214,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,131.464418,125.835409,145.2,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.006532,0.006252,0.00,0.0,0.000000,0.0,0.006126,0.000000,0.000000,C,0,0
2,blinkit_718589.0,2023-03-31,140.2,120.65,127.151290,146.810595,0.006966,0.005994,0.006317,0.007294,718589.0,blinkit,ADV-AHO-R,496.828458,0.007557,152.1,0.007557,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,137.178128,131.900206,152.1,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.006815,0.006553,0.00,0.0,0.000000,0.0,0.007214,0.006126,0.000000,C,0,0
3,blinkit_718589.0,2023-04-30,140.2,120.65,84.135918,121.411359,0.006966,0.005994,0.004180,0.006032,718589.0,blinkit,ADV-AHO-R,496.828458,0.004784,96.3,0.004784,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,93.565810,88.360591,96.3,140.2,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,140.200000,0.006966,0.0,0.0,0.0,0.004649,0.004390,0.00,0.0,0.000000,0.0,0.007557,0.007214,0.006126,C,0,0
4,blinkit_718589.0,2023-05-31,131.2,120.65,105.836774,104.297415,0.006518,0.005994,0.005258,0.005182,718589.0,blinkit,ADV-AHO-R,496.828458,0.004650,93.6,0.004650,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,114.438626,110.144987,93.6,131.2,0.0,0.0,0.0,0.0,0.000000,0.000000,-6.419401,0.000000,0.000000,0.0,131.200000,0.006518,0.0,0.0,0.0,0.005686,0.005472,0.00,0.0,0.000000,0.0,0.004784,0.007557,0.007214,C,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30457,zepto_810125.0,2026-11-30,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125.0,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-06-30,M+5,Male Grooming,0.000000,0.000000,0.0,3.6,0.0,0.0,0.0,0.0,1.493333,1.046667,141.071429,148.888889,114.285714,3.0,2.546667,0.000617,0.0,0.0,0.0,0.000000,0.000000,0.00,0.0,0.000000,0.0,0.001178,0.000562,0.000110,NPD,1,0
30458,zepto_810125.0,2026-12-31,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125.0,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-06-30,M+6,Male Grooming,0.000000,0.000000,0.0,3.6,0.0,0.0,0

In [198]:
trend_df.columns

Index(['platform_name', 'parent_material_code', 'run_month', 'trend_flag',
       'p3m_slope_flag', 'final_trend', 'lower_threshold', 'upper_threshold',
       'mean_value', 'std_value'],
      dtype='object')

In [199]:
final_df = final_df.merge(trend_df[['platform_name', 'parent_material_code', 'run_month',
                                    'final_trend','lower_threshold', 'upper_threshold']], on = ["platform_name", "parent_material_code", 'run_month'], how = 'left')
final_df


,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,blinkit_718589.0,2023-01-31,140.2,120.65,125.240172,124.714531,0.006966,0.005994,0.006222,0.006196,718589.0,blinkit,ADV-AHO-R,496.828458,0.006126,123.3,0.006126,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,135.101501,130.453417,123.3,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.006712,0.006481,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,C,0,0,1,0.013083,0.023914
1,blinkit_718589.0,2023-02-28,140.2,120.65,120.212121,131.282560,0.006966,0.005994,0.005972,0.006522,718589.0,blinkit,ADV-AHO-R,496.828458,0.007214,145.2,0.007214,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,131.464418,125.835409,145.2,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.006532,0.006252,0.00,0.0,0.000000,0.0,0.006126,0.000000,0.000000,C,0,0,1,0.013083,0.023914
2,blinkit_718589.0,2023-03-31,140.2,120.65,127.151290,146.810595,0.006966,0.005994,0.006317,0.007294,718589.0,blinkit,ADV-AHO-R,496.828458,0.007557,152.1,0.007557,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,137.178128,131.900206,152.1,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.006815,0.006553,0.00,0.0,0.000000,0.0,0.007214,0.006126,0.000000,C,0,0,1,0.013083,0.023914
3,blinkit_718589.0,2023-04-30,140.2,120.65,84.135918,121.411359,0.006966,0.005994,0.004180,0.006032,718589.0,blinkit,ADV-AHO-R,496.828458,0.004784,96.3,0.004784,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,93.565810,88.360591,96.3,140.2,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,140.200000,0.006966,0.0,0.0,0.0,0.004649,0.004390,0.00,0.0,0.000000,0.0,0.007557,0.007214,0.006126,C,0,0,1,0.013083,0.023914
4,blinkit_718589.0,2023-05-31,131.2,120.65,105.836774,104.297415,0.006518,0.005994,0.005258,0.005182,718589.0,blinkit,ADV-AHO-R,496.828458,0.004650,93.6,0.004650,2026-05-31,0.529153,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Hair Oils,114.438626,110.144987,93.6,131.2,0.0,0.0,0.0,0.0,0.000000,0.000000,-6.419401,0.000000,0.000000,0.0,131.200000,0.006518,0.0,0.0,0.0,0.005686,0.005472,0.00,0.0,0.000000,0.0,0.004784,0.007557,0.007214,C,0,0,1,0.013083,0.023914
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30457,zepto_810125.0,2026-11-30,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,810125.0,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-06-30,M+5,Male Grooming,0.000000,0.000000,0.0,3.6,0.0,0.0,0.0,0.0,1.493333,1.046667,141.071429,148.888889,114.285714,3.0,2.546667,0.000617,0.0,0.0,0.0,0.000000,0.000000,0.00,0.0,0.000000,0.0,0.001178,0.000562,0.000110,NPD,1,0,0,0.000000,0.001810
30458,zepto_810125.0,2026-12-31,0.0,0.00,0.000000,0.000000,0.00000

In [200]:
final_df = final_df.sort_values(['key', 'month_date'])

# base LY


# LY lags
final_df['ly_lag1_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(13)
)

final_df['ly_lag2_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(14)
)

# LY leads
final_df['ly_lead1_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(11)
)

final_df['ly_lead2_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(10)
)


In [201]:
# final_df['stat_bias'] = final_df['Stat Error']/final_df['Actuals Val']
# final_df = final_df.fillna(0)

# import numpy as np
# import pandas as pd

# final_df['stat_bias'] = (
#     final_df['stat_bias']
#     .replace([np.inf, -np.inf], 0)
#     .fillna(0)
# )

# bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
# labels = [
#     '< -15%',
#     '-15% to -10%',
#     '-10% to -5%',
#     '-5% to 0%',
#     '0% to 5%',
#     '5% to 10%',
#     '10% to 15%',
#     '> 15%'
# ]

# final_df['stat_bias_bucket'] = pd.cut(
#     final_df['stat_bias'],
#     bins=bins,
#     labels=labels,
#     right=False  
# )


In [202]:
final_df[final_df['month_date'] == '2026-05-31']['P3M_value'].sum()

30.905105148215103

In [203]:
brand_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'brand')
brand_seas.columns = brand_seas.columns.str.lower()
brand_seas.rename(columns={'brand':'brand_code', 'months_num':'month', 'flag':'is_seasonal_month'}, inplace=True)

final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['month'] = final_df['month_date'].dt.month
final_df = final_df.merge(brand_seas, on = ['brand_code', 'month'], how = 'left')
final_df['is_seasonal_month'].fillna(0, inplace=True)

psku_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'psku')
psku_seas.columns = psku_seas.columns.str.lower()
psku_seas.rename(columns={'months_num':'month', 'flag':'is_seasonal_month_psku'}, inplace=True)

final_df = final_df.merge(psku_seas[['parent_material_code', 'month','is_seasonal_month_psku']], on = ['parent_material_code', 'month'], how = 'left')
final_df['is_seasonal_month_psku'].fillna(0, inplace=True)
final_df['final_seasonal_month'] = np.where(
    (final_df['is_seasonal_month'] == 1) | (final_df['is_seasonal_month_psku'] == 1), 1, 0
)



In [204]:
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["final_seasonal_month"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()

adj_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',0),
        "P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum',0),
        "P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value',0),
        "P6M_non_seasonal_value": compute_adjusted_pm(x, 6,'vol_in_rum_value',0)
    })
).reset_index()
adj_df


adj_ly_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',year_shift=1),
        "LY_P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum', year_shift=1),
        "LY_P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value', year_shift=1),
        "LY_P6M_non_seasonal_value": compute_adjusted_pm(x, 6,"vol_in_rum_value", year_shift=1)
    })
).reset_index()

adj_df = adj_df.merge(adj_ly_df, on = ['key', 'run_month'], how = 'left')
#adj_df[adj_df['key'] == 'reliance_b2c_2_haryana_718488']
adj_df

,key,run_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,blinkit_718288.0,2026-06-30,63.156000,62.071000,0.877017,0.861951,51.728000,48.137000,0.718322,0.668456
1,blinkit_718310.0,2026-06-30,0.000167,0.102000,0.000006,0.003563,0.163250,0.163250,0.005702,0.005702
2,blinkit_718312.0,2026-06-30,11.068000,8.276000,0.386576,0.289059,7.406333,6.302667,0.258684,0.220136
3,blinkit_718315.0,2026-06-30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,blinkit_718317.0,2026-06-30,0.000000,0.000000,0.000000,0.000000,0.000000,0.100000,0.000000,0.000004
...,...,...,...,...,...,...,...,...,...,...
898,zepto_810685.0,2026-06-30,0.000667,0.009333,0.000021,0.000294,NaN,NaN,NaN,NaN
899,zepto_810738.0,2026-06-30,114.512667,116.986333,0.004197,0.004287,NaN,NaN,NaN,NaN
900,zepto_810971.0,2026-06-30,0.600000,0.600000,0.000772,0.000772,NaN,NaN,NaN,NaN
901,zepto_811005.0,2026-06-30,0.564667,0.564667,0.000726,0.000726,NaN,NaN,NaN,NaN


In [205]:
final_df.shape

(30462, 68)

In [206]:
# adj_df.to_csv('seasonal_p3m_qcom.csv', index=False)
#all[all['month_date'].isin(['2025-11-30','2025-12-31','2026-01-31'])].groupby(['key','run_month','month_date','final_seasonal_month'])['vol_in_rum'].sum().reset_index().to_csv('seasonal_month_check.csv', index=False)
final_df = final_df.merge(
    adj_df,
    on=['key'],
    how="left"
)
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month_x,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,blinkit_718288.0,2023-01-31,21.828,21.337833,14.141480,23.283180,0.303115,0.296308,0.196376,0.323322,718288.0,blinkit,SAFF GOLD,138865.260689,0.313613,22.584,0.313613,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,16.919146,15.494658,22.584,0.000000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.234948,0.215167,0.000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,A,0,0,0,0.617524,1.138415,NaN,NaN,NaN,NaN,1,0.0,NaN,0.0,0,2026-06-30,63.156000,62.071000,0.877017,0.861951,51.728,48.137,0.718322,0.668456
1,blinkit_718288.0,2023-02-28,21.828,21.337833,15.747385,21.731667,0.303115,0.296308,0.218676,0.301777,718288.0,blinkit,SAFF GOLD,138865.260689,0.243958,17.568,0.243958,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,18.337736,17.084799,17.568,0.000000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.254647,0.237249,0.000,0.0,0.000000,0.0,0.313613,0.000000,0.000000,A,0,0,0,0.617524,1.138415,NaN,NaN,NaN,NaN,2,0.0,NaN,0.0,0,2026-06-30,63.156000,62.071000,0.877017,0.861951,51.728,48.137,0.718322,0.668456
2,blinkit_718288.0,2023-03-31,21.828,21.337833,26.868887,25.585143,0.303115,0.296308,0.373116,0.355289,718288.0,blinkit,SAFF GOLD,138865.260689,0.351773,25.332,0.351773,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,29.713462,28.386028,25.332,0.000000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.412617,0.394183,0.000,0.0,0.000000,0.0,0.243958,0.313613,0.000000,A,0,0,0,0.617524,1.138415,NaN,NaN,NaN,NaN,3,0.0,NaN,0.0,0,2026-06-30,63.156000,62.071000,0.877017,0.861951,51.728,48.137,0.718322,0.668456
3,blinkit_718288.0,2023-04-30,21.828,21.337833,10.384195,23.536022,0.303115,0.296308,0.144200,0.326834,718288.0,blinkit,SAFF GOLD,138865.260689,0.303948,21.888,0.303948,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,13.267045,11.706763,21.888,21.828000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0,0.0,21.828000,0.303115,0.0,0.0,0.0,0.184233,0.162566,0.000,0.0,0.000000,0.0,0.351773,0.243958,0.313613,A,0,0,0,0.617524,1.138415,NaN,NaN,NaN,NaN,4,0.0,NaN,0.0,0,2026-06-30,63.156000,62.071000,0.877017,0.861951,51.728,48.137,0.718322,0.668456
4,blinkit_718288.0,2023-05-31,21.596,21.337833,12.988040,21.183205,0.299893,0.296308,0.180359,0.294161,718288.0,blinkit,SAFF GOLD,138865.260689,0.220240,15.860,0.220240,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,15.667556,14.121985,15.860,21.596000,0.0,0.0,0.0,0.0,0.0000,0.0000,-1.062855,0.000000,0.0,0.0,21.596000,0.299893,0.0,0.0,0.0,0.217568,0.196105,0.0

In [207]:
all_brand = final_df.groupby(['brand_code', 'run_month_x','month_date'])['vol_in_rum_value'].sum().reset_index()
def detect_month_anomaly(df, brand_code, month_num, threshold=0.25, months_window=3):
    """
    Detect if a specific month's vol_in_rum_value is >25% different 
    from past 3 months & next 3 months, and if pattern repeats in last 2 years.
    
    Parameters:
    - df: input dataframe with 'month_date', 'vol_in_rum_value'
    - brand_code: filter by this brand code
    - month_num: month to check (6, 7, 8, 9)
    - threshold: 25% difference threshold
    - months_window: number of months before and after to compare
    """
    
    df_brand = df[df['brand_code'] == brand_code].sort_values('month_date').copy()
    
    if df_brand.empty:
        return None
    
    df_brand['year'] = df_brand['month_date'].dt.year
    df_brand['month'] = df_brand['month_date'].dt.month
    
    years = sorted(df_brand['year'].unique())
    current_year = years[-1]
    past_years = [y for y in years if y < current_year][-2:]
    
    anomalies = []
    
    for year in past_years:
        df_year = df_brand[df_brand['year'] == year].sort_values('month_date')
        
        month_data = df_year[df_year['month'] == month_num]
        if month_data.empty:
            continue
        
        month_value = month_data['vol_in_rum_value'].iloc[0]
        
        past_months = [(month_num - i - 1) % 12 + 1 for i in range(1, months_window + 1)]
        next_months = [(month_num + i - 1) % 12 + 1 for i in range(1, months_window + 1)]
        
        past_m = df_year[df_year['month'].isin(past_months)]['vol_in_rum_value']
        next_m = df_year[df_year['month'].isin(next_months)]['vol_in_rum_value']
        
        comparison_values = pd.concat([past_m])
        
        if comparison_values.empty:
            continue
        
        pct_diffs = []
        for comp_value in comparison_values:
            if comp_value != 0:
                pct_diff = (month_value - comp_value) / comp_value
                pct_diffs.append(pct_diff)
        
        if pct_diffs:
            positive_diffs = [p for p in pct_diffs if p > 0]
            negative_diffs = [p for p in pct_diffs if p < 0]
            same_sign = len(positive_diffs) == len(pct_diffs) or len(negative_diffs) == len(pct_diffs)
            is_anomaly = len([p for p in pct_diffs if abs(p) > threshold]) == len(pct_diffs) and same_sign
        else:
            is_anomaly = False

        anomalies.append({
            'brand_code': brand_code,
            'month': month_num,
            'year': year,
            'month_value': month_value,
            'num_months_compared': len(comparison_values),
            'pct_diffs_from_each': pct_diffs,
            'min_pct_diff': min(pct_diffs) * 100 if pct_diffs else None,
            'max_pct_diff': max(pct_diffs) * 100 if pct_diffs else None,
            'is_anomaly': is_anomaly,
            'direction': 'higher' if month_value > comparison_values.mean() else 'lower'
        })
    
    if len(anomalies) == 2:
        pattern_repeats = anomalies[0]['is_anomaly'] and anomalies[1]['is_anomaly']
        return pd.DataFrame(anomalies), pattern_repeats
    
    return pd.DataFrame(anomalies), False


# Check months 6, 7, 8, 9
brands = all_brand['brand_code'].unique()
results = []

for month in [6, 7, 8, 9]:
    for brand in brands:
        df_result, repeats = detect_month_anomaly(all_brand, brand, month)
        if df_result is not None and not df_result.empty:
            df_result['pattern_repeats'] = repeats
            results.append(df_result)

anomaly_summary = pd.concat(results, ignore_index=True)
print(anomaly_summary[anomaly_summary['pattern_repeats'] == True])

    brand_code  month  year  month_value  num_months_compared  \
12   H&C_ALMND      6  2025     0.006731                    3   
13   H&C_ALMND      6  2026     0.000000                    3   
17     KAYA_GM      6  2025     0.072632                    3   
18     KAYA_GM      6  2026     0.000000                    3   
21      KERALA      6  2025     0.000486                    3   
..         ...    ...   ...          ...                  ...   
524  SAF_PNBTR      8  2026     0.000000                    3   
531  SFFT_VNGR      8  2025     0.000000                    3   
532  SFFT_VNGR      8  2026     0.000000                    3   
560   SW NOGAS      8  2025     0.000211                    3   
561   SW NOGAS      8  2026     0.000000                    3   

                                   pct_diffs_from_each  min_pct_diff  \
12   [0.779596977329975, 0.5112299465240643, 0.4374...     43.743642   
13                                  [-1.0, -1.0, -1.0]   -100.000000   
17 

In [208]:
# mnth = 6
# all_brand = final_df.groupby(['brand_code', 'run_month_x','month_date'])['vol_in_rum_value'].sum().reset_index()
# def detect_april_anomaly(df, brand_code, threshold=0.25, months_window=3):
#     """
#     Detect if April's vol_in_rum_value is >25% different 
#     from past 3 months & next 3 months, and if pattern repeats in last 2 years.
    
#     Parameters:
#     - df: input dataframe with 'month_date', 'vol_in_rum_value', 'run_month'
#     - brand_code: filter by this brand code
#     - threshold: 25% difference threshold
#     - months_window: number of months before and after April to compare
    
#     """
    
#     # Filter for brand and sort by month_date
#     df_brand = df[df['brand_code'] == brand_code].sort_values('month_date').copy()
    
#     if df_brand.empty:
#         return None
    
#     # Extract year and month
#     df_brand['year'] = df_brand['month_date'].dt.year
#     df_brand['month'] = df_brand['month_date'].dt.month
    
#     # Get unique years (excluding current year if incomplete)
#     years = sorted(df_brand['year'].unique())
#     current_year = years[-1]
#     past_years = [y for y in years if y < current_year][-2:]  # Last 2 years
    
#     anomalies = []
    
#     # Check each past year's April
#     for year in past_years:
#         df_year = df_brand[df_brand['year'] == year].sort_values('month_date')
        
#         # Get April data (month == 4)
#         april_data = df_year[df_year['month'] == mnth]
#         if april_data.empty:
#             continue
        
#         april_value = april_data['vol_in_rum_value'].iloc[0]
#         april_month = mnth
        
#         # Dynamically calculate past and next months
#         past_months = [(april_month - i - 1) % 12 + 1 for i in range(1,months_window+1)]
#         print(past_months)
#         next_months = [(april_month + i - 1) % 12 + 1 for i in range(1, months_window + 1)]
#         print(next_months)
        
#         # Get past and next months values
#         past_3m = df_year[df_year['month'].isin(past_months)]['vol_in_rum_value']
#         next_3m = df_year[df_year['month'].isin(next_months)]['vol_in_rum_value']
        
#         # Combine all comparison months
#         comparison_values = pd.concat([past_3m, next_3m])
        
#         if comparison_values.empty:
#             continue
        
#         # Calculate mean of comparison months
#         #mean_value = comparison_values.mean()
        
#         # Calculate percentage difference
#         pct_diffs = []
#         for comp_value in comparison_values:
#             if comp_value != 0:
#                 pct_diff = (april_value - comp_value) / comp_value
#                 pct_diffs.append(pct_diff)
        
#         # April is anomalous if it's >25% different from ALL comparison months
#         # AND all differences have the same sign (all positive or all negative)
#         if pct_diffs:
#             positive_diffs = [p for p in pct_diffs if p > 0]
#             negative_diffs = [p for p in pct_diffs if p < 0]
#             same_sign = len(positive_diffs) == len(pct_diffs) or len(negative_diffs) == len(pct_diffs)
#             is_anomaly = len([p for p in pct_diffs if abs(p) > threshold]) == len(pct_diffs) and same_sign
#         else:
#             is_anomaly = False

#         anomalies.append({
#             'brand_code': brand_code,
#             'year': year,
#             'april_value': april_value,
#             'num_months_compared': len(comparison_values),
#             'pct_diffs_from_each': pct_diffs,
#             'min_pct_diff': min(pct_diffs) * 100 if pct_diffs else None,
#             'max_pct_diff': max(pct_diffs) * 100 if pct_diffs else None,
#             'is_anomaly': is_anomaly,
#             'direction': 'higher' if april_value > comparison_values.mean() else 'lower'
#         })
    
#     # Check if pattern repeats in both years
#     if len(anomalies) == 2:
#         pattern_repeats = anomalies[0]['is_anomaly'] and anomalies[1]['is_anomaly']
#         return pd.DataFrame(anomalies), pattern_repeats
    
#     return pd.DataFrame(anomalies), False


# # Usage: Apply to each brand code
# brands = all_brand['brand_code'].unique()
# results = []

# for brand in brands:
#     df_result, repeats = detect_april_anomaly(all_brand, brand)
#     if df_result is not None and not df_result.empty:
#         df_result['pattern_repeats'] = repeats
#         results.append(df_result)


# anomaly_summary = pd.concat(results, ignore_index=True)
# print(anomaly_summary[anomaly_summary['pattern_repeats'] == True])

In [209]:
final_df.shape

(30462, 77)

In [210]:
final_brands = anomaly_summary[anomaly_summary['pattern_repeats'] == True].drop_duplicates(subset=['brand_code','month'])[['brand_code', 'month','direction', 'min_pct_diff', 'max_pct_diff']]
final_brands['month_different'] = 1
final_df['month'] = final_df['month_date'].dt.month
final_df = final_df.merge(final_brands[['brand_code', 'month','month_different']], on = ['brand_code','month'], how = 'left')
final_df['month_different'].fillna(0, inplace=True)
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month_x,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,class,skipped,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value,month_different
0,blinkit_718288.0,2023-01-31,21.828,21.337833,14.141480,23.283180,0.303115,0.296308,0.196376,0.323322,718288.0,blinkit,SAFF GOLD,138865.260689,0.313613,22.584,0.313613,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,16.919146,15.494658,22.584,0.000000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.234948,0.215167,0.000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,A,0,0,0,0.617524,1.138415,NaN,NaN,NaN,NaN,1,0.0,NaN,0.0,0,2026-06-30,63.156000,62.071000,0.877017,0.861951,51.728,48.137,0.718322,0.668456,0.0
1,blinkit_718288.0,2023-02-28,21.828,21.337833,15.747385,21.731667,0.303115,0.296308,0.218676,0.301777,718288.0,blinkit,SAFF GOLD,138865.260689,0.243958,17.568,0.243958,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,18.337736,17.084799,17.568,0.000000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.254647,0.237249,0.000,0.0,0.000000,0.0,0.313613,0.000000,0.000000,A,0,0,0,0.617524,1.138415,NaN,NaN,NaN,NaN,2,0.0,NaN,0.0,0,2026-06-30,63.156000,62.071000,0.877017,0.861951,51.728,48.137,0.718322,0.668456,0.0
2,blinkit_718288.0,2023-03-31,21.828,21.337833,26.868887,25.585143,0.303115,0.296308,0.373116,0.355289,718288.0,blinkit,SAFF GOLD,138865.260689,0.351773,25.332,0.351773,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,29.713462,28.386028,25.332,0.000000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.412617,0.394183,0.000,0.0,0.000000,0.0,0.243958,0.313613,0.000000,A,0,0,0,0.617524,1.138415,NaN,NaN,NaN,NaN,3,0.0,NaN,0.0,0,2026-06-30,63.156000,62.071000,0.877017,0.861951,51.728,48.137,0.718322,0.668456,0.0
3,blinkit_718288.0,2023-04-30,21.828,21.337833,10.384195,23.536022,0.303115,0.296308,0.144200,0.326834,718288.0,blinkit,SAFF GOLD,138865.260689,0.303948,21.888,0.303948,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,13.267045,11.706763,21.888,21.828000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0,0.0,21.828000,0.303115,0.0,0.0,0.0,0.184233,0.162566,0.000,0.0,0.000000,0.0,0.351773,0.243958,0.313613,A,0,0,0,0.617524,1.138415,NaN,NaN,NaN,NaN,4,0.0,NaN,0.0,0,2026-06-30,63.156000,62.071000,0.877017,0.861951,51.728,48.137,0.718322,0.668456,0.0
4,blinkit_718288.0,2023-05-31,21.596,21.337833,12.988040,21.183205,0.299893,0.296308,0.180359,0.294161,718288.0,blinkit,SAFF GOLD,138865.260689,0.220240,15.860,0.220240,2026-05-31,0.431175,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None,Saffola Oils,15.667556,14.121985,15.860,21.596000,0.0,0.0,0.0,0.0,0.0000,0.0000,-1.062855,0.000000,0.0,0.0,21.596000,0.299893,0

In [221]:
final_df[final_df['month_date'] == '2026-07-31']['pred_value_prophet'].sum()

31.695867376151153

In [223]:
final_df.to_csv('/data/aman_singh/acuuracy_check/all_combination_qcom_trend_cp.csv')

In [222]:
final_df[(final_df['M month'].notna())].to_csv('/data/aman_singh/acuuracy_check/qcom_chain_psku_july_pred.csv')